# A1 -Testupplägg och läckageskydd

In [ ]:
from datetime import date
from pathlib import Path

import pandas as pd


from elpris.splits import available_at_noon, make_targets, split_origins, valid_origins



DATA_DIR = Path("../data")
daily = pd.read_csv(DATA_DIR / "processed" / "se3_daily.csv", parse_dates=["date"])
daily["date"] = daily["date"].dt.date
daily.tail()

,date,mean,max,min
1802,2026-10-08,1.291076,1.976380,0.305912
1803,2026-10-09,0.338855,0.763910,0.151062
1804,2026-10-10,0.358729,0.801017,0.143662
1805,2026-10-11,0.885049,1.590175,0.453885
1806,2026-10-12,1.404023,2.215412,0.313758


## 1. Vilken data var kända kl. 12?

In [11]:
for origin in [date(2026, 10, 5), date(2020, 1, 1)]:
    known = available_at_noon(daily, origin)
    print(origin, "→", known["date"].max(), len(known))

2026-10-05 → 2026-10-05 1800
2020-01-01 → nan 0


## 2. Targets
```
origin = 5 okt
target:   6 okt   7 okt   8 okt   9 okt   10 okt   11 okt   12 okt
horizon:    1       2       3       4       5        6        7
```

In [12]:
make_targets(daily, date(2026, 10, 5))

,origin_date,target_date,horizon,mean,max,min
0,2026-10-05,2026-10-06,1,0.416642,1.357273,-0.015405
1,2026-10-05,2026-10-07,2,1.199681,1.777942,0.868845
2,2026-10-05,2026-10-08,3,1.291076,1.976380,0.305912
3,2026-10-05,2026-10-09,4,0.338855,0.763910,0.151062
4,2026-10-05,2026-10-10,5,0.358729,0.801017,0.143662
5,2026-10-05,2026-10-11,6,0.885049,1.590175,0.453885
6,2026-10-05,2026-10-12,7,1.404023,2.215412,0.313758


```
 available_at_noon(D)  →  modellen  →  y_pred
 make_targets(D)       →  endast facit  →  y_true
                          MAE = |y_true − y_pred|
```

## 3. Giltiga prognostillfällen

In [13]:
origins = valid_origins(daily)
print(len(origins), origins[0], origins[-1])

1794 2021-11-07 2026-10-05


## 4. Tidsuppdelning

```
 train origins ... │ gap (7 dagar) │ test origins ...
                      origin+7 < test_start

```

In [14]:
TEST_START = date(2026, 4, 1)  # Förslag – beslutas i gruppen

train_origins, test_origins = split_origins(origins, TEST_START)
print("Train:", len(train_origins), train_origins[0], train_origins[-1])
print("Test:", len(test_origins), test_origins[0], test_origins[-1])

Train: 1599 2021-11-07 2026-03-24
Test: 188 2026-04-01 2026-10-05
